# Cosine Similarity Demo using Latitude & Longitude

> **Teaching Note:** This notebook intentionally uses latitude/longitude as simple 2D vectors to explain cosine similarity. Later discuss that cosine similarity measures **direction**, not true geographic distance.


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown

locations = {
    "Delhi": (28.6139,77.2090),
    "Noida": (28.5355,77.3910),
    "Gurgaon": (28.4595,77.0266),
    "Kanpur": (26.4499,80.3319),
    "Lucknow": (26.8467,80.9462),
    "Jaipur": (26.9124,75.7873),
    "Mumbai": (19.0760,72.8777),
    "Pune": (18.5204,73.8567),
    "Ahmedabad": (23.0225,72.5714),
    "Hyderabad": (17.3850,78.4867),
    "Bangalore": (12.9716,77.5946),
    "Chennai": (13.0827,80.2707),
    "Kolkata": (22.5726,88.3639),
    "Bhopal": (23.2599,77.4126),
    "Patna": (25.5941,85.1376),
    "London": (51.5074, -0.1278),
    "Paris": (48.8566, 2.3522),
    "Berlin": (52.5200, 13.4050),
    "Moscow": (55.7558, 37.6173),
    "Dubai": (25.2048, 55.2708),
    "Singapore": (1.3521, 103.8198),
    "Tokyo": (35.6762, 139.6503),
    "Beijing": (39.9042, 116.4074),
    "Sydney": (-33.8688, 151.2093),
    "New York": (40.7128, -74.0060),
    "San Francisco": (37.7749, -122.4194),
    "Toronto": (43.6532, -79.3832),
}

df = pd.DataFrame(
    [(k,v[0],v[1]) for k,v in locations.items()],
    columns=["City","Latitude","Longitude"]
)
print(f"2D Vector = <Latitude, Longitude>")
df

2D Vector = <Latitude, Longitude>


,City,Latitude,Longitude
0,Delhi,28.6139,77.2090
1,Noida,28.5355,77.3910
2,Gurgaon,28.4595,77.0266
3,Kanpur,26.4499,80.3319
4,Lucknow,26.8467,80.9462
5,Jaipur,26.9124,75.7873
6,Mumbai,19.0760,72.8777
7,Pune,18.5204,73.8567
8,Ahmedabad,23.0225,72.5714
9,Hyderabad,17.3850,78.4867


In [7]:
def cosine(v1,v2):
    v1=np.array(v1,float)
    v2=np.array(v2,float)
    return np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))

def geo_distance(v1,v2):
    lat1,lon1=np.radians(v1)
    lat2,lon2=np.radians(v2)
    dlat=lat2-lat1
    dlon=lon2-lon1
    a=np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371*2*np.arcsin(np.sqrt(a))

def demo(city1,city2):
    v1=locations[city1]
    v2=locations[city2]

    cos=cosine(v1,v2)
    dist=geo_distance(v1,v2)

    plt.figure(figsize=(8,8))
    plt.scatter(df.Longitude,df.Latitude,s=60)

    for _,r in df.iterrows():
        plt.text(r.Longitude+0.2,r.Latitude+0.2,r.City,fontsize=8)

    plt.arrow(0,0,v1[1],v1[0],head_width=1,length_includes_head=True)
    plt.arrow(0,0,v2[1],v2[0],head_width=1,length_includes_head=True)

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Latitude/Longitude treated as 2D vectors")
    plt.grid(True)
    # plt.xlim(0,95)
    # plt.ylim(0,35)
    plt.xlim(df.Longitude.min() - 10, df.Longitude.max() + 10)
    plt.ylim(df.Latitude.min() - 10, df.Latitude.max() + 10)
    plt.show()

    print(f"City 1 : {city1}")
    print(f"Vector : {v1}")
    print()
    print(f"City 2 : {city2}")
    print(f"Vector : {v2}")
    print()
    print(f"Cosine Similarity : {cos:.6f}")
    print(f"Geographic Distance (km): {dist:.1f}")
    print()
    print("Discussion:")
    print("- High cosine => vectors point in similar directions.")
    print("- Geographic distance is shown separately.")
    print("- Later, embeddings use the SAME cosine mathematics,")
    print("  but vectors represent semantic meaning instead of geography.")

interact(
    demo,
    city1=Dropdown(options=sorted(locations.keys()),value="Delhi"),
    city2=Dropdown(options=sorted(locations.keys()),value="Kanpur")
);

interactive(children=(Dropdown(description='city1', index=6, options=('Ahmedabad', 'Bangalore', 'Beijing', 'Be…

#### Similarity Search
* **Question:** Find the top 5 cities that are closest to the given city

In [13]:
city_names = list(locations.keys())
city_vectors = np.array(list(locations.values()), dtype=float)

# Normalize
city_vectors = city_vectors / np.linalg.norm(city_vectors, axis=1, keepdims=True)
print("===== Position Vectors ====")
pd.DataFrame(zip(city_names, np.round(city_vectors, decimals=4))).head()

===== Position Vectors ====


,0,1
0,Delhi,"[0.3475, 0.9377]"
1,Noida,"[0.346, 0.9383]"
2,Gurgaon,"[0.3466, 0.938]"
3,Kanpur,"[0.3127, 0.9498]"
4,Lucknow,"[0.3148, 0.9492]"


In [ ]:
def similarity_search(city_name, top_k=5):
    """
    Returns the most similar cities using cosine similarity.
    """

    if city_name not in locations:
        print(f"Unknown city: {city_name}")
        return

    query = np.array(locations[city_name], dtype=float)
    query = query / np.linalg.norm(query)

    scores = city_vectors @ query

    ranking = np.argsort(scores)[::-1]

    print(f"\nMost similar cities to '{city_name}'\n")
    print("-" * 55)

    count = 0

    for idx in ranking:

        candidate = city_names[idx]

        if candidate == city_name:
            continue

        print(f"{candidate:20s}  Cosine = {scores[idx]:.6f}")

        count += 1

        if count >= top_k:
            break

interact(
    similarity_search,
    city_name=Dropdown(options=sorted(locations.keys()),value="Delhi")
);

interactive(children=(Dropdown(description='city_name', index=6, options=('Ahmedabad', 'Bangalore', 'Beijing',…

## Classroom Discussion

Try:

- Delhi vs Noida
- Delhi vs Mumbai
- Delhi vs Chennai
- Bangalore vs Chennai
- Kolkata vs Jaipur


> **Question:** Does cosine similarity measure physical distance?

> Answer:
**No.** It measures the angle (direction) between vectors. Embedding vectors use the same mathematics, but the dimensions represent semantic features rather than latitude and longitude.
>


# 🤔 FAQ: Why is Beijing More Similar to Delhi than Lucknow?

After running the similarity search, we may observe something surprising:

| Query City | Similar City | Cosine Similarity |
|------------|--------------|------------------:|
| Delhi | Gurgaon | ~1.000000 |
| Delhi | Noida | ~0.999999 |
| Delhi | Jaipur | ~0.999906 |
| **Delhi** | **Beijing** | **~0.999696** |
| Delhi | Lucknow | ~0.999399 |

At first glance, this appears incorrect because **Lucknow is geographically much closer to Delhi than Beijing is.**

So why does Beijing receive a higher cosine similarity score?

---

## Cosine Similarity Measures Direction, Not Distance

The vectors used in this notebook are simply:

```python
(latitude, longitude)
```

For example,

```text
Delhi      = (28.61, 77.20)
Lucknow    = (26.85, 80.95)
Beijing    = (39.90,116.40)
```

Cosine similarity **does not measure how far apart two points are.**

Instead, it measures **how closely two vectors point in the same direction from the origin (0°, 0°).**

Mathematically,

$$
\text{Cosine Similarity} =
\frac{A \cdot B}{||A||\,||B||}
$$

Only the **angle** between the vectors matters.

---

## Visual Intuition

Imagine the Earth projected onto a graph.

```
                     Beijing
                        ●
                       /
                      /
                     /
                    /
Delhi ●------------/

        \
         \
          \
           ● Lucknow
```

Although Lucknow is much closer to Delhi geographically, the vector from the origin to **Beijing** happens to point in almost the same direction as the vector to **Delhi**.

Cosine similarity only sees this angle.

---

## Does This Mean They Are in the Same Time Zone?

No.

This is a common misconception.

In fact,

- India uses **UTC +5:30**
- China uses **UTC +8:00**

They are **not** in the same time zone.

The higher cosine similarity is simply a mathematical consequence of the latitude and longitude values having a similar direction from the origin.

---

## Why Doesn't This Represent Geographic Closeness?

Latitude and longitude were designed to represent **physical locations on Earth**, not semantic similarity.

For measuring real-world distance, we should instead use metrics such as:

- Euclidean Distance (approximate)
- Haversine Distance (great-circle distance)
- Vincenty's Formula (high precision)

These correctly identify Lucknow as being much closer to Delhi than Beijing.

---

# 🚀 The Real Purpose of This Demo

This notebook intentionally uses **latitude and longitude** only to explain **how cosine similarity works mathematically**.

The important takeaway is:

- **Cosine Similarity compares vector direction.**
- **It does not compare physical distance.**

Tomorrow, instead of latitude and longitude, we will use **embedding vectors**.

Unlike GPS coordinates, embeddings are **learned by a neural network** so that vectors pointing in similar directions represent **similar meanings**.

For example,

| Sentence | Embedding |
|-----------|-----------|
| Employees receive 18 vacation days. | → 384-dimensional vector |
| Annual leave policy allows 18 days. | → 384-dimensional vector |

These vectors point in nearly the same direction because they have **similar semantic meaning**, not because they share similar words.

The mathematics (cosine similarity) remains **exactly the same**.

Only the **meaning of the vectors changes**.

---

## 💡 Key Takeaway

> **Cosine Similarity is not a measure of distance.**
>
> It is a measure of **how similarly two vectors point**.
>
> In semantic search and RAG systems, the vectors are embeddings that represent **meaning**, making cosine similarity an excellent way to retrieve semantically related documents.